In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score, accuracy_score, f1_score
from imblearn.over_sampling import RandomOverSampler
import joblib

In [2]:
# model initiate
model_name = "logistic_reg"
seed = 20240627
n_splits = 5
lr = LogisticRegression(max_iter=5000, random_state=seed)
scoring_funcs = {
    'precision': precision_score,
    'recall': recall_score,
    'roc_auc': roc_auc_score,
    'accuracy': accuracy_score,
    'f1': f1_score
}

In [3]:
# cross validation function
def run_cross_validation(model_id, use_upsampling=False):
    # load data
    X = pd.read_csv(f"data/X_train_{model_id}.csv")
    y = pd.read_csv(f"data/y_train_{model_id}.csv").values.ravel()
    # set k fold 
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_results = {metric: [] for metric in scoring_funcs.keys()}

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Oversampling is only performed on the training set.
        if use_upsampling:
            sampler = RandomOverSampler(random_state=seed)
            X_train, y_train = sampler.fit_resample(X_train, y_train)
        
        # train model
        lr.fit(X_train, y_train)
        y_pred = lr.predict(X_val)
        y_proba = lr.predict_proba(X_val)[:, 1]
        
        # calculate score and save into fold_results
        for metric_name, metric_func in scoring_funcs.items():
            # only roc_auc use y_pred
            score = metric_func(y_val, y_proba if metric_name == 'roc_auc' else y_pred)
            fold_results[metric_name].append(score)
    # convert fold_result to numpy        
    final_scores = {k: np.array(v) for k, v in fold_results.items()}
    
    # print result and save
    print(model_id)
    print(f"Mean Recall: {final_scores['recall'].mean():.3f}")
    print(f"Mean ROC: {final_scores['roc_auc'].mean():.3f}")
    print(f"Mean Accuracy: {final_scores['accuracy'].mean():.3f}")

    # save result
    save_path = f"results/{model_id}_{model_name}_cv_results.pkl"
    joblib.dump(final_scores, save_path)


In [4]:
# set oversampling config
model_configs = {
    "model_1": False,
    "model_1a": True,
    "model_1b": True,
    "model_2": False,
    "model_2a": True,
    "model_2b": True
}

# train cross validation
for model_id, need_upsample in model_configs.items():
    run_cross_validation(model_id, use_upsampling=need_upsample)

model_1
Mean Recall: 0.817
Mean ROC: 0.871
Mean Accuracy: 0.794
model_1a
Mean Recall: 0.764
Mean ROC: 0.821
Mean Accuracy: 0.729
model_1b
Mean Recall: 0.779
Mean ROC: 0.836
Mean Accuracy: 0.768
model_2
Mean Recall: 0.865
Mean ROC: 0.775
Mean Accuracy: 0.727
model_2a
Mean Recall: 0.693
Mean ROC: 0.730
Mean Accuracy: 0.669
model_2b
Mean Recall: 0.726
Mean ROC: 0.781
Mean Accuracy: 0.718
